# Preprocessing — Predicting Daily Citi Bike Ridership

**Notebook 2 of 3.** Here I fix the problems found in EDA and build the features the model
needs. At the end I save the cleaned data to `data/citibike_weather_daily_clean.csv` —
that's the only file `model.ipynb` will use.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/citibike_weather_daily.csv')
df.dtypes

ride_date               str
num_rides             int64
avg_duration_min    float64
temp_f              float64
max_temp_f          float64
min_temp_f          float64
wind_speed_knots    float64
precip_in           float64
day_of_week             str
month                 int64
dtype: object

## C1: Fix the Dtypes

EDA found one dtype problem: `ride_date` loaded as a string. I convert it to a real datetime.
Every other column loaded correctly, so nothing else needs converting.

In [2]:
df['ride_date'] = pd.to_datetime(df['ride_date'])

print(df['ride_date'].dtype)
print('First day:', df['ride_date'].min())
print('Last day: ', df['ride_date'].max())

datetime64[us]
First day: 2013-07-01 00:00:00
Last day:  2018-05-31 00:00:00


## C2: Handle the Coded Missing Value

EDA found one sentinel: `precip_in = 99.99` (NOAA's code for "missing") on 2016-02-11.

Step 1: turn the code into a real `NaN` so it stops pretending to be data.
Step 2: decide whether to drop the row or impute a value.

In [3]:
df['precip_in'] = df['precip_in'].replace(99.99, np.nan)

print('Missing values per column:')
print(df.isna().sum())

Missing values per column:
ride_date           0
num_rides           0
avg_duration_min    0
temp_f              0
max_temp_f          0
min_temp_f          0
wind_speed_knots    0
precip_in           1
day_of_week         0
month               0
dtype: int64


In [4]:
# My decision: impute 0.0 (reasons in the cell below)
df['precip_in'] = df['precip_in'].fillna(0.0)

print('Missing values left:', df.isna().sum().sum())
print('precip_in max is now:', df['precip_in'].max())

Missing values left: 0
precip_in max is now: 4.88


### Why I imputed 0.0 instead of dropping the row

- Only **1 row out of 1,610** is affected, and everything else in that row (rides, temps,
  wind, calendar) is good data. Dropping the whole row would throw away a valid day just to
  avoid estimating one cell.
- **Why 0.0:** the median precipitation in this dataset is exactly 0.0 (most NYC days are
  dry), and the days right around Feb 11, 2016 recorded 0.00–0.06 inches — it was a dry cold
  week. So both a "typical value" imputation and a "nearby day" imputation point to
  approximately 0.
- I did NOT use the mean, because the mean of precipitation is pulled up by a few big storm
  days (and was inflated by the sentinel itself) — it wouldn't represent a normal day.

## C3: Encode Day of Week

The model can't multiply 'Tuesday' by a coefficient, so I one-hot encode `day_of_week` into
0/1 indicator columns.

In [5]:
dow_dummies = pd.get_dummies(df['day_of_week'], prefix='dow', drop_first=True)
df = pd.concat([df, dow_dummies], axis=1)

print(list(dow_dummies.columns))
df.head(3)

['dow_Monday', 'dow_Saturday', 'dow_Sunday', 'dow_Thursday', 'dow_Tuesday', 'dow_Wednesday']


,ride_date,num_rides,avg_duration_min,temp_f,max_temp_f,min_temp_f,wind_speed_knots,precip_in,day_of_week,month,dow_Monday,dow_Saturday,dow_Sunday,dow_Thursday,dow_Tuesday,dow_Wednesday
0,2013-07-01,16650,16.309988,74.8,78.1,73.4,7.8,0.00,Monday,7,True,False,False,False,False,False
1,2013-07-02,22745,15.968826,76.1,82.9,73.0,8.0,0.73,Tuesday,7,False,False,False,False,True,False
2,2013-07-03,21864,16.238808,78.5,84.9,73.9,8.8,0.06,Wednesday,7,False,False,False,False,False,True


### Why I used drop_first=True

If I keep all 7 day columns, the model has a redundancy problem: the 7 columns always add up
to 1, so any 6 of them already tell you the 7th. That's called the "dummy variable trap."
Dropping one day fixes it, and it makes the coefficients easy to read: the dropped day
becomes the baseline. `drop_first` drops the alphabetically-first day, **Friday**, so every
`dow_` coefficient means "compared to a Friday with the same weather".

## C4: Build a Trend Feature

EDA (E4) showed the system roughly doubled from 2013 to 2017, so the model needs to know
what year it is. I build both options from the assignment and pick one.

In [6]:
df['year'] = df['ride_date'].dt.year
df['days_since_launch'] = (df['ride_date'] - df['ride_date'].min()).dt.days

df[['ride_date', 'year', 'days_since_launch']].head()

,ride_date,year,days_since_launch
0,2013-07-01,2013,0
1,2013-07-02,2013,1
2,2013-07-03,2013,2
3,2013-07-04,2013,3
4,2013-07-05,2013,4


### My choice: days_since_launch

I'll use **`days_since_launch`** in the model. A plain `year` column is coarse — it jumps
once every January 1st and stays flat for 12 months. `days_since_launch` grows a little every
day, which matches the smooth growth I saw in the E4 plot. I keep the `year` column in the
file anyway in case it's useful.

## C5 (Stretch): Extra Features

Each one is tied to something I actually saw in EDA:

In [7]:
# Squared temperature: E3 showed rides rise with temperature but flatten past ~85F.
# A squared term lets a linear model fit a curve instead of a straight line.
df['temp_sq'] = df['temp_f'] ** 2

# Weekend flag: E5 showed Saturday and Sunday are ~20-25% quieter than weekdays.
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# Rained-at-all flag: E3 showed rainy days sit lower, and going from no rain to some rain
# matters more than going from 1.0 to 1.3 inches.
df['rained'] = (df['precip_in'] > 0).astype(int)

df[['temp_f', 'temp_sq', 'is_weekend', 'rained']].head()

,temp_f,temp_sq,is_weekend,rained
0,74.8,5595.04,0,0
1,76.1,5791.21,0,1
2,78.5,6162.25,0,1
3,82.0,6724.00,0,1
4,84.4,7123.36,0,0


## C6: Save the Clean Dataset

One last thing before saving: I drop `avg_duration_min`. **Rule 1** says every feature must
be knowable before the day starts, and average trip duration is calculated from the very
rides we're trying to predict. Dropping it here means it can't accidentally leak into the
model later.

I keep all three temperature columns in the *file* — Rule 2 is about what goes into the
*model*, and `model.ipynb` will pick just one.

In [8]:
df = df.drop(columns=['avg_duration_min'])

df.to_csv('../data/citibike_weather_daily_clean.csv', index=False)

# Reload it to double-check it saved correctly
check = pd.read_csv('../data/citibike_weather_daily_clean.csv')
print('Saved:', check.shape)
print(list(check.columns))

Saved: (1610, 20)
['ride_date', 'num_rides', 'temp_f', 'max_temp_f', 'min_temp_f', 'wind_speed_knots', 'precip_in', 'day_of_week', 'month', 'dow_Monday', 'dow_Saturday', 'dow_Sunday', 'dow_Thursday', 'dow_Tuesday', 'dow_Wednesday', 'year', 'days_since_launch', 'temp_sq', 'is_weekend', 'rained']


### Preprocessing summary

- `ride_date`: string → datetime
- `precip_in`: 99.99 sentinel → NaN → imputed 0.0
- `day_of_week`: one-hot encoded (Friday is the baseline)
- Added `year` and `days_since_launch` (trend), plus `temp_sq`, `is_weekend`, `rained`
- Dropped `avg_duration_min` (Rule 1)
- Saved everything to `citibike_weather_daily_clean.csv` — the raw file's job is done